# 📚 Chi-Square & Categorical Tests: Testing Counts, Not Means
> Section 8 of the Statistics & A/B Testing curriculum
> 통계 & A/B 테스트 커리큘럼 섹션 8

---
# 🎯 Learning Objective (학습 목표)
Today I want to learn / 오늘 배우고 싶은 것:
- [x] Run a chi-square goodness-of-fit test to check one categorical variable against an expected distribution<br>카이제곱 적합도 검정으로 하나의 범주형 변수를 기대 분포와 비교하기
- [x] Run a chi-square test of independence to check whether two categorical variables are related<br>카이제곱 독립성 검정으로 두 범주형 변수가 관련이 있는지 확인하기
- [x] Check the expected-cell-count assumption and switch to Fisher's exact test when it's violated<br>기대빈도 가정을 확인하고 위반 시 Fisher의 정확검정으로 전환하기

---
# 🧠 Concept (개념)

## What is it? (이것은 무엇인가?)
*(Explain it in your own words.)*

**EN:** Chi-square tests are the categorical-data counterpart to t-tests: instead of comparing means of continuous numbers, they compare COUNTS across categories. The goodness-of-fit test asks "does this one categorical variable's distribution match an expected pattern?", while the test of independence asks "are two categorical variables related to each other, or independent?"

**KR:** 카이제곱 검정은 t-검정의 범주형 데이터 버전입니다: 연속형 숫자의 평균을 비교하는 대신, 범주별 개수(count)를 비교합니다. 적합도 검정은 "이 하나의 범주형 변수의 분포가 기대되는 패턴과 일치하는가?"를 묻고, 독립성 검정은 "두 범주형 변수가 서로 관련이 있는가, 아니면 독립인가?"를 묻습니다.

### Two chi-square test types (두 가지 카이제곱 검정 유형)

| | Goodness of Fit / 적합도 | Test of Independence / 독립성 |
|---|---|---|
| Question / 질문 | Does ONE variable's distribution match an expected pattern?<br>하나의 변수 분포가 기대 패턴과 일치하는가? | Are TWO variables related?<br>두 변수가 관련이 있는가? |
| Data shape / 데이터 형태 | One categorical variable, multiple categories<br>범주형 변수 하나, 여러 범주 | Two categorical variables, cross-tabulated<br>범주형 변수 둘, 교차표 |
| Business example / 예시 | Is our channel mix still 40/35/25?<br>채널 구성이 여전히 40/35/25인가? | Does device type relate to purchase category?<br>기기 종류가 구매 카테고리와 관련 있는가? |
| scipy function | `stats.chisquare()` | `stats.chi2_contingency()` |

## Why do we use it? (왜 사용하는가?)
*(When is it useful?)*

**EN:** Most of the statistics covered so far (means, t-tests, correlation) work on numbers. But a huge share of business data is categorical: traffic source, device type, plan tier, product category. Chi-square tests let you formally test claims about this kind of data — "is our traffic mix still matching last year's split?", "does device type actually relate to what people buy?" — with the same H0/p-value logic from Section 6.

**KR:** 지금까지 다룬 대부분의 통계(평균, t-검정, 상관관계)는 숫자에 적용됩니다. 하지만 비즈니스 데이터의 상당 부분은 범주형입니다: 트래픽 소스, 기기 종류, 플랜 등급, 제품 카테고리. 카이제곱 검정은 이런 종류의 데이터에 대한 주장을 섹션 6의 같은 H0/p-value 논리로 공식적으로 검정하게 해줍니다 -- "우리 트래픽 구성이 작년 비율과 여전히 일치하는가?", "기기 종류가 실제로 구매 품목과 관련이 있는가?" 

## When is it used in Business Analytics? (비즈니스 분석에서 언제 사용되는가?)
*(Real-world use case)*

**EN:** Checking whether this month's channel mix matches a target allocation (goodness of fit), testing whether a marketing segment relates to conversion behavior (independence), analyzing survey responses broken down by demographic group, and quality-checking whether an A/B test's traffic split actually landed at the intended ratio (goodness of fit again, this time as a "Sample Ratio Mismatch" check — a concept that reappears directly in the A/B testing sections later).

**KR:** 이번 달 채널 구성이 목표 배분과 일치하는지 확인(적합도), 마케팅 세그먼트가 전환 행동과 관련 있는지 검정(독립성), 인구통계 그룹별 설문 응답 분석, 그리고 A/B 테스트의 트래픽 분할이 실제로 의도한 비율로 나왔는지 품질 확인(다시 적합도, 이번엔 "표본 비율 불일치" 확인 — 이 커리큘럼 후반 A/B 테스트 섹션에서 직접 다시 등장하는 개념).

---
# 📝 Syntax (문법)

## Basic Syntax (기본 문법)

In [9]:
import numpy as np
import pandas as pd
from scipy import stats

# Goodness of fit: does this month's traffic source mix match the historical split?
# 적합도: 이번 달 트래픽 소스 구성이 과거 비율과 일치하는가?
observed = np.array([420, 310, 270])  # Organic, Paid, Referral (observed counts)
expected_proportions = np.array([0.40, 0.35, 0.25])  # historical split
total = observed.sum()
expected = expected_proportions * total

chi2_stat, p_value = stats.chisquare(f_obs=observed, f_exp=expected)

print(f"Observed: {observed}")
print(f"Expected: {expected}")
print(f"Chi-square statistic: {chi2_stat:.3f}")
print(f"p-value: {p_value:.4f}")

Observed: [420 310 270]
Expected: [400. 350. 250.]
Chi-square statistic: 7.171
p-value: 0.0277


## Common Variations (자주 쓰는 변형)

In [10]:
# Test of independence: are two categorical variables related?
# 독립성 검정: 두 범주형 변수가 관련이 있는가?
contingency_table = pd.DataFrame({
    'Mobile': [180, 90],
    'Desktop': [70, 160],
}, index=['Bought Basic', 'Bought Premium'])

chi2_stat, p_value, dof, expected_table = stats.chi2_contingency(contingency_table)

print(f"Chi-square statistic: {chi2_stat:.3f}")
print(f"p-value: {p_value:.6f}")
print(f"Degrees of freedom: {dof}")

Chi-square statistic: 63.776
p-value: 0.000000
Degrees of freedom: 1


---
# 🧪 Small Examples (예제)

## Example 1 — Goodness of Fit: Does This Month's Channel Mix Match the Target? (이번 달 채널 구성이 목표와 일치하는가?)

In [11]:
# Business question: leadership set a target channel mix of 40% Organic,
# 35% Paid, 25% Referral. Does this month's ACTUAL mix match that target?
# 비즈니스 질문: 경영진이 목표 채널 구성을 오가닉 40%, 페이드 35%,
# 리퍼럴 25%로 정했습니다. 이번 달 실제 구성이 그 목표와 일치하는가?

observed = np.array([510, 340, 250])  # Organic, Paid, Referral
labels = ['Organic', 'Paid', 'Referral']
expected_proportions = np.array([0.40, 0.35, 0.25])
total = observed.sum()
expected = expected_proportions * total

chi2_stat, p_value = stats.chisquare(f_obs=observed, f_exp=expected)

print("Channel   Observed   Expected")
for label, obs, exp in zip(labels, observed, expected):
    print(f"{label:9s} {obs:8d}   {exp:8.1f}")
print()
print(f"Chi-square statistic: {chi2_stat:.3f}")
print(f"p-value: {p_value:.4f}")
print()
if p_value < 0.05:
    print("p < 0.05 -> the observed mix is significantly different from the target --")
    print("worth investigating WHICH channel(s) are furthest off.")
    print("p < 0.05 -> 관측된 구성이 목표와 유의미하게 다름 -- 어느 채널이 가장")
    print("벗어나 있는지 조사할 가치가 있습니다.")
else:
    print("p >= 0.05 -> no strong evidence the mix has drifted from the target.")
    print("p >= 0.05 -> 구성이 목표에서 벗어났다는 강한 증거 없음.")

Channel   Observed   Expected
Organic        510      440.0
Paid           340      385.0
Referral       250      275.0

Chi-square statistic: 18.669
p-value: 0.0001

p < 0.05 -> the observed mix is significantly different from the target --
worth investigating WHICH channel(s) are furthest off.
p < 0.05 -> 관측된 구성이 목표와 유의미하게 다름 -- 어느 채널이 가장
벗어나 있는지 조사할 가치가 있습니다.


## Example 2 — Chi-Square Test of Independence: Device Type and Purchase Category (기기 종류와 구매 카테고리)

In [12]:
# Business question: does device type (mobile vs desktop) relate to WHAT
# category of product people buy, or are the two independent?
# 비즈니스 질문: 기기 종류(모바일 vs 데스크톱)가 사람들이 구매하는
# 제품 카테고리와 관련이 있는가, 아니면 두 변수가 독립인가?

contingency = pd.DataFrame({
    'Mobile': [320, 140, 90],
    'Desktop': [110, 210, 180],
}, index=['Apparel', 'Electronics', 'Home Goods'])

print("Contingency table (observed counts):")
print(contingency)
print()

chi2_stat, p_value, dof, expected = stats.chi2_contingency(contingency)

print(f"Chi-square statistic: {chi2_stat:.3f}")
print(f"Degrees of freedom: {dof}")
print(f"p-value: {p_value:.2e}")
print()
if p_value < 0.05:
    print("p < 0.05 -> device type and purchase category are significantly related --")
    print("mobile shoppers and desktop shoppers buy meaningfully different mixes.")
    print("p < 0.05 -> 기기 종류와 구매 카테고리가 유의미하게 관련됨 -- 모바일")
    print("쇼핑객과 데스크톱 쇼핑객은 의미 있게 다른 구성으로 구매합니다.")

Contingency table (observed counts):
             Mobile  Desktop
Apparel         320      110
Electronics     140      210
Home Goods       90      180

Chi-square statistic: 144.505
Degrees of freedom: 2
p-value: 4.18e-32

p < 0.05 -> device type and purchase category are significantly related --
mobile shoppers and desktop shoppers buy meaningfully different mixes.
p < 0.05 -> 기기 종류와 구매 카테고리가 유의미하게 관련됨 -- 모바일
쇼핑객과 데스크톱 쇼핑객은 의미 있게 다른 구성으로 구매합니다.


## Example 3 — Expected vs Observed: Reading a Contingency Table (분할표 읽기)

In [13]:
# Business question: given the significant result from Example 2, WHICH
# specific combinations of device x category are driving the difference most?
# 비즈니스 질문: 예제 2의 유의미한 결과가 나왔을 때, 기기×카테고리의
# 어느 조합이 그 차이를 가장 크게 이끄는가?

chi2_stat, p_value, dof, expected = stats.chi2_contingency(contingency)
expected_df = pd.DataFrame(expected, index=contingency.index, columns=contingency.columns)

print("Observed counts:")
print(contingency)
print("\nExpected counts (if device and category were independent):")
print(expected_df.round(1))

# Per-cell contribution to the chi-square statistic
# 각 셀의 카이제곱 통계량 기여도
contribution = (contingency - expected_df) ** 2 / expected_df
print("\nEach cell's contribution to the chi-square statistic:")
print(contribution.round(2))

biggest_cell = contribution.stack().idxmax()
print(f"\nBiggest driver of the difference: {biggest_cell[0]} x {biggest_cell[1]}")
print(f"(observed={contingency.loc[biggest_cell[0], biggest_cell[1]]}, "
      f"expected={expected_df.loc[biggest_cell[0], biggest_cell[1]]:.1f})")
print(f"차이의 가장 큰 요인: {biggest_cell[0]} x {biggest_cell[1]}")

Observed counts:
             Mobile  Desktop
Apparel         320      110
Electronics     140      210
Home Goods       90      180

Expected counts (if device and category were independent):
             Mobile  Desktop
Apparel       225.2    204.8
Electronics   183.3    166.7
Home Goods    141.4    128.6

Each cell's contribution to the chi-square statistic:
             Mobile  Desktop
Apparel       39.87    43.85
Electronics   10.24    11.27
Home Goods    18.70    20.57

Biggest driver of the difference: Apparel x Desktop
(observed=110, expected=204.8)
차이의 가장 큰 요인: Apparel x Desktop


## Example 4 — Assumptions & Sample Size: When to Use Fisher's Exact Test (Fisher의 정확검정을 써야 할 때)

In [14]:
# Business question: a brand-new premium feature just launched to a small
# beta group. Does usage relate to customer segment? With such a SMALL
# sample, is chi-square still reliable?
# 비즈니스 질문: 신규 프리미엄 기능이 소규모 베타 그룹에 막 출시됐습니다.
# 사용 여부가 고객 세그먼트와 관련이 있는가? 이렇게 작은 표본에서
# 카이제곱이 여전히 신뢰할 만한가?

small_table = pd.DataFrame({
    'Used feature': [8, 2],
    'Did not use': [4, 9],
}, index=['Power users', 'Casual users'])

print("Small sample contingency table:")
print(small_table)

chi2_stat, p_value, dof, expected = stats.chi2_contingency(small_table)
expected_df = pd.DataFrame(expected, index=small_table.index, columns=small_table.columns)

print("\nExpected counts:")
print(expected_df.round(2))
print(f"\nSmallest expected cell count: {expected_df.values.min():.2f}")

if expected_df.values.min() < 5:
    print("At least one expected cell count is below 5 -- the standard chi-square")
    print("approximation becomes unreliable here. Use Fisher's exact test instead.")
    print("기대빈도가 5 미만인 셀이 있습니다 -- 표준 카이제곱 근사가 여기서는")
    print("신뢰할 수 없습니다. 대신 Fisher의 정확검정을 사용하세요.")
    print()
    odds_ratio, fisher_p = stats.fisher_exact(small_table)
    print(f"Fisher's exact test: odds ratio={odds_ratio:.2f}, p-value={fisher_p:.4f}")
else:
    print("All expected cell counts are >= 5 -- chi-square should be reliable here.")

print(f"\n(For comparison, the standard chi-square gave p={p_value:.4f} -- these can")
print("disagree meaningfully when the sample is this small.)")

Small sample contingency table:
              Used feature  Did not use
Power users              8            4
Casual users             2            9

Expected counts:
              Used feature  Did not use
Power users           5.22         6.78
Casual users          4.78         6.22

Smallest expected cell count: 4.78
At least one expected cell count is below 5 -- the standard chi-square
approximation becomes unreliable here. Use Fisher's exact test instead.
기대빈도가 5 미만인 셀이 있습니다 -- 표준 카이제곱 근사가 여기서는
신뢰할 수 없습니다. 대신 Fisher의 정확검정을 사용하세요.

Fisher's exact test: odds ratio=9.00, p-value=0.0361

(For comparison, the standard chi-square gave p=0.0546 -- these can
disagree meaningfully when the sample is this small.)


## Example 5 — Practice: Apply All Four Concepts (연습: 네 가지 개념 모두 적용하기)

Fill in every `________` blank below. The cell runs with **no errors** even if you leave blanks — it just prints a reminder instead of that part's result.

아래 `________` 빈칸을 모두 채워보세요. 빈칸을 두어도 셀은 **에러 없이** 실행되며, 그 부분의 결과 대신 안내 메시지만 출력됩니다.

In [15]:
# ============================================================
# Practice -- fill in every ________ blank, then re-run the cell
# 연습 -- 아래 ________ 빈칸을 모두 채운 뒤 셀을 다시 실행하세요
# ============================================================
BLANK = "________"

def ready(*vals):
    return all(v != BLANK for v in vals)

# ---------- Part A: goodness of fit -- survey response distribution ----------
observed_a = np.array([145, 210, 95, 50])  # Very Satisfied, Satisfied, Neutral, Dissatisfied
expected_props_a = np.array([0.30, 0.40, 0.20, 0.10])  # historical/expected split

test_func_a = 'chisquare'   # TODO: which scipy.stats function tests ONE variable's distribution
                      # against expected proportions? ('chisquare')

if ready(test_func_a):
    total_a = observed_a.sum()
    expected_a = expected_props_a * total_a
    chi2_a, p_a = getattr(stats, test_func_a)(f_obs=observed_a, f_exp=expected_a)
    print(f"Chi-square: {chi2_a:.3f}, p-value: {p_a:.4f}")
else:
    print("Part A \u270f\ufe0f  Fill in test_func_a above, then re-run.")

# ---------- Part B: test of independence -- region and subscription plan ----------
table_b = pd.DataFrame({
    'Basic': [120, 90, 60],
    'Premium': [80, 130, 140],
}, index=['West', 'Central', 'East'])

test_func_b = 'chi2_contingency'   # TODO: which scipy.stats function tests whether TWO categorical
                      # variables are related? ('chi2_contingency')

if ready(test_func_b):
    chi2_b, p_b, dof_b, exp_b = getattr(stats, test_func_b)(table_b)
    print(f"Chi-square: {chi2_b:.3f}, dof: {dof_b}, p-value: {p_b:.6f}")
else:
    print("Part B \u270f\ufe0f  Fill in test_func_b above, then re-run.")

# ---------- Part C: expected frequency -- compute by hand ----------
row_total_c = 200    # total for a given row
col_total_c = 350    # total for a given column
grand_total_c = 800  # total of the whole table

expected_cell_c = (row_total_c * col_total_c) / grand_total_c   # TODO: apply the expected-frequency formula:
                          # (row_total * col_total) / grand_total

if ready(expected_cell_c):
    print(f"Expected count for this cell: {expected_cell_c:.2f}")
else:
    print("Part C \u270f\ufe0f  Fill in expected_cell_c above, then re-run.")

# ---------- Part D: assumption check -- decide chi-square vs Fisher's ----------
tiny_table_d = pd.DataFrame({
    'Converted': [3, 1],
    'Did not convert': [7, 9],
}, index=['Group A', 'Group B'])
_, _, _, expected_d = stats.chi2_contingency(tiny_table_d)

use_fisher_d = True   # TODO: True or False -- given the small counts here, should you
                       # use Fisher's exact test instead of standard chi-square?

if ready(use_fisher_d):
    min_expected = expected_d.min()
    print(f"Smallest expected cell count: {min_expected:.2f}")
    correct = (min_expected < 5) == use_fisher_d
    print(f"Your choice: use Fisher's = {use_fisher_d}")
    if correct:
        print("Correct!")
        print("정답입니다!")
    else:
        print("Not quite -- check whether the smallest expected count is below 5.")
        print("다시 확인해보세요 -- 가장 작은 기대빈도가 5 미만인지 확인하세요.")
else:
    print("Part D \u270f\ufe0f  Fill in use_fisher_d above (True or False), then re-run.")

Chi-square: 0.917, p-value: 0.8214
Chi-square: 37.576, dof: 2, p-value: 0.000000
Expected count for this cell: 87.50
Smallest expected cell count: 2.00
Your choice: use Fisher's = True
Correct!
정답입니다!


---
# ⚠️ Common Mistakes (자주 하는 실수)

### Mistake 1 — Running Chi-Square on Raw Numeric Data Instead of Counts (원본 숫자 데이터에 카이제곱 사용)
Chi-square tests work on FREQUENCY COUNTS within categories, not on continuous measurements — trying to feed it raw order amounts or ages directly (instead of, say, counts of people in age BUCKETS) is a category error.   
카이제곱 검정은 범주 내 빈도(개수)에 대해 작동하지, 연속형 측정값에는 작동하지 않습니다 -- 원본 주문금액이나 나이를 (연령 구간별 개수가 아니라) 그대로 넣으려는 것은 범주 자체를 잘못 이해한 것입니다.

✅ **Fix / 해결:** Bucket continuous data into categories first if needed, and always confirm your input is a count/frequency table, not raw values.

### Mistake 2 — Ignoring the "Expected Count >= 5" Rule (기대빈도 5 이상 규칙 무시)
When any expected cell count falls below 5 (common with rare categories or small samples), the chi-square distribution stops approximating the test statistic well, and the p-value becomes unreliable — exactly the situation Example 4 demonstrated.  
어떤 셀의 기대빈도든 5 미만이면(드문 범주나 작은 표본에서 흔함), 카이제곱 분포가 검정통계량을 더 이상 잘 근사하지 못해 p-value가 신뢰할 수 없게 됩니다 -- 예제 4가 보여준 바로 그 상황입니다.

✅ **Fix / 해결:** Always check the expected frequencies (not the observed ones) before trusting a chi-square p-value; switch to Fisher's exact test when any expected cell is below 5.

### Mistake 3 — Treating "Not Significant" as "Definitely No Relationship" ("유의하지 않음"을 "관계가 확실히 없음"으로 취급)
Just like with t-tests (Section 7), a non-significant chi-square result might reflect low statistical power (too few observations) rather than proving the two variables are truly unrelated.  
t-검정(섹션 7)과 마찬가지로, 유의하지 않은 카이제곱 결과는 두 변수가 정말 관련이 없다는 증거라기보다 낮은 검정력(관측치가 너무 적음)을 반영하는 것일 수 있습니다.

✅ **Fix / 해결:** Check the sample size and expected counts before treating a "not significant" chi-square result as a final, confident answer.

### Mistake 4 — Feeding Percentages Instead of Raw Counts (원본 개수 대신 백분율 입력)
Chi-square's math is sensitive to the actual sample size behind each number — a table of percentages (which always sum to 100 regardless of how many people were surveyed) throws away that information and produces a meaningless or misleading result.  
카이제곱의 계산은 각 숫자 뒤에 있는 실제 표본 크기에 민감합니다 -- 백분율 표(몇 명을 조사했든 항상 합이 100이 되는)는 그 정보를 버려서 무의미하거나 오해를 부르는 결과를 만듭니다.

✅ **Fix / 해결:** Always feed `chisquare()` or `chi2_contingency()` raw observed COUNTS, never pre-computed percentages.

### Mistake 5 — Assuming a Significant Result Tells You WHICH Cells Drive It (유의미한 결과가 어느 셀이 원인인지 알려준다고 착각)
A significant chi-square test of independence only tells you THAT an association exists somewhere in the table — it doesn't by itself point to which specific category combination is most responsible.  
유의미한 카이제곱 독립성 검정은 표 어딘가에 연관성이 존재한다는 것만 알려줍니다 -- 그 자체로는 어느 특정 범주 조합이 가장 큰 원인인지 짚어주지 않습니다.

✅ **Fix / 해결:** Follow up a significant result with a per-cell contribution breakdown (Example 3's approach) to find the specific drivers.

### Mistake 6 — Using Chi-Square on the SAME Subjects Measured Twice (같은 대상을 두 번 측정한 데이터에 카이제곱 사용)
Chi-square assumes each observation is independent — using it to compare "before" and "after" category counts from the SAME group of people violates that assumption, exactly like using an independent t-test on paired data (Section 7's Mistake 1).  
카이제곱은 각 관측치가 독립이라고 가정합니다 -- 같은 사람들 그룹의 "전"과 "후" 범주 개수를 비교하는 데 사용하면 이 가정을 위반합니다. 대응 데이터에 독립표본 t-검정을 쓰는 것과 정확히 같은 문제입니다(섹션 7의 실수 1).

✅ **Fix / 해결:** For the SAME subjects' categorical responses measured twice, use a paired categorical test (like McNemar's test) instead of standard chi-square.

---
# 💡 Tips (팁)
Useful tips or shortcuts / 유용한 팁과 단축법

- `pd.crosstab()` builds a contingency table directly from two DataFrame columns in one line — pair it with `stats.chi2_contingency()` for the fastest path from raw data to a test result.<br>pd.crosstab()은 DataFrame의 두 컬럼에서 한 줄로 분할표를 만듭니다 -- stats.chi2_contingency()와 짝지으면 원본 데이터에서 검정 결과까지 가장 빠른 경로입니다.
- A 2x2 contingency table (two categories each) is the categorical equivalent of comparing two proportions with a z-test — both approaches usually agree closely.<br>2x2 분할표(각각 두 범주)는 두 비율을 z-검정으로 비교하는 것의 범주형 버전입니다 -- 두 접근법은 보통 서로 밀접하게 일치합니다.
- `stats.chi2_contingency()` returns FOUR things in this order: `(chi2_stat, p_value, degrees_of_freedom, expected_table)` — the expected table alone is often as useful as the p-value for spotting where a difference lives.<br>stats.chi2_contingency()는 이 순서로 네 가지를 반환합니다: (chi2_stat, p_value, degrees_of_freedom, expected_table) -- 기대빈도 표 자체가 차이가 어디 있는지 찾는 데 p-value만큼 유용한 경우가 많습니다.
- Larger contingency tables (more rows/columns) need proportionally more total observations to keep every expected cell above 5 — a 2x2 table is far more forgiving of a small sample than a 5x5 one.<br>더 큰 분할표(행·열이 많은)는 모든 기대빈도를 5 이상으로 유지하려면 비례적으로 더 많은 총 관측치가 필요합니다 -- 2x2 표는 5x5 표보다 작은 표본에 훨씬 관대합니다.
- Chi-square answers "is there a relationship?" the same way correlation (Section 5) does for numeric data — but it doesn't tell you the DIRECTION or STRENGTH the way a correlation coefficient does; consider Cramér's V for that.<br>카이제곱은 숫자 데이터에 대한 상관관계(섹션 5)와 같은 방식으로 "관계가 있는가?"에 답합니다 -- 하지만 상관계수처럼 관계의 방향이나 강도를 알려주지는 않습니다 -- 이를 위해서는 Cramér's V를 고려하세요.

---
# 🔗 Related Concepts (관련 개념)

```
t-Tests (Section 7)
   one-sample / independent / paired, checking assumptions
        ↓
Chi-Square & Categorical Tests (Section 8)  ← you are here / 현재 위치
   goodness of fit, test of independence, contingency tables, Fisher's exact test
        ↓
ANOVA (Section 9) -> Non-Parametric Tests (Section 10)
        ↓
[ Part 3: Regression ] -> [ Part 4: A/B Testing in Practice ]
```

*How is today's topic connected to other concepts?*

**EN:** The chi-square test of independence from Example 2 is doing for two CATEGORICAL variables exactly what Section 5's correlation coefficient did for two NUMERIC variables — testing whether they move together. And just like Section 5's correlation-isn't-causation warning, a significant chi-square association between device type and purchase category doesn't prove device type CAUSES the purchase pattern; a confounder (like age, which correlates with both) could still be responsible. The goodness-of-fit test also reappears directly in A/B testing as the "Sample Ratio Mismatch" check — verifying your traffic actually split the way you intended before trusting any other result from the experiment.

**KR:** 예제 2의 카이제곱 독립성 검정은 두 범주형 변수에 대해, 섹션 5의 상관계수가 두 수치형 변수에 대해 했던 것과 정확히 같은 일을 합니다 -- 함께 움직이는지 검정하는 것입니다. 그리고 섹션 5의 "상관관계는 인과관계가 아니다" 경고와 마찬가지로, 기기 종류와 구매 카테고리 사이의 유의미한 카이제곱 연관성이 기기 종류가 구매 패턴을 유발한다는 것을 증명하지는 않습니다 -- 둘 다와 상관된 교란변수(나이 등)가 여전히 원인일 수 있습니다. 적합도 검정은 A/B 테스트에서 "표본 비율 불일치(Sample Ratio Mismatch)" 확인으로 직접 다시 등장합니다 -- 실험의 다른 결과를 신뢰하기 전에 트래픽이 실제로 의도한 대로 분할되었는지 검증하는 것입니다.

---
# 💼 Business Example (비즈니스 예시)
*How would a Business Analyst use this?*

**Scenario / 시나리오:**

**EN:** Before trusting the results of a live A/B test, your team runs two quality checks: (1) did the 50/50 traffic split between Control and Treatment actually land close to 50/50 (a Sample Ratio Mismatch check), and (2) is there any unexpected relationship between which variant a user saw and which DEVICE they used (which would suggest the randomization itself was flawed)?

**KR:** 진행 중인 A/B 테스트의 결과를 신뢰하기 전에, 팀은 두 가지 품질 점검을 실행합니다: (1) 대조군과 실험군 사이의 50/50 트래픽 분할이 실제로 50/50에 가깝게 나왔는가(표본 비율 불일치 확인), (2) 사용자가 본 변형과 사용한 기기 사이에 예상치 못한 관계가 있는가(있다면 무작위 배정 자체에 결함이 있음을 시사).

**To-do / 할 일:**
- [x] Run a goodness-of-fit test checking the traffic split against the intended 50/50<br>적합도 검정으로 트래픽 분할이 의도한 50/50과 일치하는지 확인하기
- [x] Run a test of independence checking variant assignment against device type<br>독립성 검정으로 변형 배정과 기기 종류를 확인하기
- [x] Decide whether either QA check raises a red flag before trusting the main A/B result<br>본 A/B 결과를 신뢰하기 전에 품질 점검에서 위험 신호가 있는지 판단하기
- [x] Write a go/no-go recommendation for trusting the experiment's results<br>실험 결과 신뢰 여부에 대한 진행/보류 제안 작성하기

In [16]:
np.random.seed(30)

# QA check 1: Sample Ratio Mismatch -- is the split close to the intended 50/50?
n_total = 5000
observed_split = np.array([2560, 2440])  # Control, Treatment
expected_split = np.array([0.5, 0.5]) * n_total

srm_chi2, srm_p = stats.chisquare(f_obs=observed_split, f_exp=expected_split)

print("QA Check 1: Sample Ratio Mismatch (traffic split)")
print(f"  Observed: Control={observed_split[0]}, Treatment={observed_split[1]}")
print(f"  Expected: 50/50 = {expected_split[0]:.0f}/{expected_split[1]:.0f}")
print(f"  Chi-square p-value: {srm_p:.4f}")
if srm_p < 0.01:  # SRM checks conventionally use a stricter threshold
    print("  FLAG: split deviates significantly from 50/50 -- investigate randomization")
    print("  경고: 분할이 50/50에서 유의미하게 벗어남 -- 무작위 배정 조사 필요")
else:
    print("  OK: split is consistent with the intended 50/50 ratio")
    print("  정상: 분할이 의도한 50/50 비율과 일치함")

# QA check 2: does variant assignment relate to device type? (it shouldn't, if randomized properly)
variant_device = pd.DataFrame({
    'Mobile': [1650, 1600],
    'Desktop': [910, 840],
}, index=['Control', 'Treatment'])

indep_chi2, indep_p, dof, expected = stats.chi2_contingency(variant_device)

print()
print("QA Check 2: Variant assignment vs device type")
print(variant_device)
print(f"  Chi-square p-value: {indep_p:.4f}")
if indep_p < 0.05:
    print("  FLAG: variant assignment is related to device type -- randomization may be flawed")
    print("  경고: 변형 배정이 기기 종류와 관련됨 -- 무작위 배정에 결함이 있을 수 있음")
else:
    print("  OK: no evidence variant assignment relates to device type")
    print("  정상: 변형 배정이 기기 종류와 관련 있다는 증거 없음")

print()
print("Recommendation: both QA checks pass -- proceed with trusting the main A/B test result.")
print("제안: 두 품질 점검 모두 통과 -- 본 A/B 테스트 결과를 신뢰하고 진행해도 됩니다.")

QA Check 1: Sample Ratio Mismatch (traffic split)
  Observed: Control=2560, Treatment=2440
  Expected: 50/50 = 2500/2500
  Chi-square p-value: 0.0897
  OK: split is consistent with the intended 50/50 ratio
  정상: 분할이 의도한 50/50 비율과 일치함

QA Check 2: Variant assignment vs device type
           Mobile  Desktop
Control      1650      910
Treatment    1600      840
  Chi-square p-value: 0.4233
  OK: no evidence variant assignment relates to device type
  정상: 변형 배정이 기기 종류와 관련 있다는 증거 없음

Recommendation: both QA checks pass -- proceed with trusting the main A/B test result.
제안: 두 품질 점검 모두 통과 -- 본 A/B 테스트 결과를 신뢰하고 진행해도 됩니다.


---
# 📝 Summary (요약)
*Write today's concept in 3~5 sentences.*

**EN:** Chi-square tests are the categorical counterpart to t-tests: instead of comparing means of numbers, they compare counts within categories. The goodness-of-fit test checks whether ONE categorical variable's observed distribution matches an expected pattern (like verifying a channel mix hits a target, or that an A/B test's traffic split landed at the intended ratio), while the test of independence checks whether TWO categorical variables are related — the categorical equivalent of Section 5's correlation, complete with its own version of the correlation-isn't-causation warning. Both tests rely on comparing observed counts to expected counts in a contingency table, and both require a minimum expected cell count of 5 to trust the standard approximation; when that assumption breaks (small samples, rare categories), Fisher's exact test is the more reliable alternative.

**KR:** 카이제곱 검정은 t-검정의 범주형 버전입니다: 숫자의 평균을 비교하는 대신 범주 내 개수를 비교합니다. 적합도 검정은 하나의 범주형 변수의 관측된 분포가 기대되는 패턴과 일치하는지 확인하고(채널 구성이 목표에 도달했는지, A/B 테스트의 트래픽 분할이 의도한 비율로 나왔는지 검증 등), 독립성 검정은 두 범주형 변수가 관련이 있는지 확인합니다 -- 섹션 5의 상관관계에 대응하는 범주형 버전이며, 상관관계-인과관계 아님 경고의 자체 버전도 함께 갖습니다. 두 검정 모두 분할표에서 관측 개수와 기대 개수를 비교하는 데 의존하며, 표준 근사를 신뢰하려면 최소 기대빈도 5가 필요합니다 -- 이 가정이 깨지면(작은 표본, 드문 범주) Fisher의 정확검정이 더 신뢰할 만한 대안입니다.

---
# 📌 One Sentence Summary (한 문장 요약)
Today's topic in ONE sentence. / 오늘의 주제를 한 문장으로.

> **EN:** Chi-square tests compare category COUNTS the way t-tests compare means — goodness of fit checks one variable against an expected pattern, independence checks two variables against each other — and both need every expected cell count to be at least 5, falling back to Fisher's exact test when it isn't.

> **KR:** 카이제곱 검정은 t-검정이 평균을 비교하듯 범주별 개수를 비교합니다 -- 적합도는 하나의 변수를 기대 패턴과, 독립성은 두 변수를 서로 비교하며 -- 둘 다 모든 기대빈도가 최소 5는 되어야 하고, 그렇지 않으면 Fisher의 정확검정으로 대체합니다.

---
# ❓ Review Questions (복습 질문)

**Q1 (EN).** A company wants to know if this quarter's device mix (mobile/desktop/tablet) matches last year's split. Which chi-square test fits, and why?  
**Q1 (KR).** 어떤 회사가 이번 분기 기기 구성(모바일/데스크톱/태블릿)이 작년 비율과 일치하는지 알고 싶어합니다. 어떤 카이제곱 검정이 맞으며, 이유는 무엇인가요?

A goodness-of-fit test fits: one categorical variable's observed device mix is compared with last year's proportions.  
적합도 검정이 맞습니다. 범주 변수 하나의 관측된 기기 구성을 작년 비율과 비교합니다.

**Q2 (EN).** Why can't you run a chi-square test directly on a table of percentages instead of raw counts?  
**Q2 (KR).** 원본 개수 대신 백분율 표에 카이제곱 검정을 직접 돌릴 수 없는 이유는 무엇인가요?

The test uses counts, which carry the sample size, and 5 of 10 is not the same evidence as 500 of 1,000.
검정은 표본 크기가 담긴 개수를 쓰며, 10명 중 5명과 1,000명 중 500명은 증거의 크기가 다릅니다.

**Q3 (EN).** A chi-square test of independence comes back significant. Does that tell you WHICH specific category combination is driving the association? What would you do to find out?  
**Q3 (KR).** 카이제곱 독립성 검정 결과가 유의미하게 나왔습니다. 이것이 어느 특정 범주 조합이 그 연관성을 이끄는지 알려주나요? 알아내려면 무엇을 해야 하나요?

No. It only says an association exists; check each cell's standardized residuals to see which cells drive it.  
아니요. 연관이 있다는 것만 알려 주며, 셀별 표준화 잔차를 보면 어느 셀이 이끄는지 알 수 있습니다.

**Q4 (EN).** What does it mean when an expected cell count in a contingency table is below 5, and what should you do instead of trusting the standard chi-square p-value?  
**Q4 (KR).** 분할표의 기대빈도가 5 미만이라는 것은 무엇을 의미하며, 표준 카이제곱 p-value를 신뢰하는 대신 무엇을 해야 하나요?

Expected counts below 5 mean too few observations for the chi-square approximation, so use Fisher's exact test or merge categories.  
기대빈도 5 미만은 카이제곱 근사를 쓰기에 관측이 너무 적다는 뜻이라, Fisher 정확 검정을 쓰거나 범주를 합칩니다.

**Q5 (EN).** Why is a goodness-of-fit test a natural way to check for a "Sample Ratio Mismatch" in an A/B test's traffic split?  
**Q5 (KR).** 적합도 검정이 A/B 테스트 트래픽 분할의 "표본 비율 불일치"를 확인하는 자연스러운 방법인 이유는 무엇인가요?

The planned split, such as 50:50, gives the expected proportions, and a goodness-of-fit test compares the actual user counts with them.  
계획한 분할(예: 50:50)이 기대 비율이 되고, 적합도 검정으로 실제 사용자 수를 그 비율과 비교합니다.

---
*📅 Try answering these again in a few days. / 며칠 후 다시 답해보세요.*